# 02. Optimización y Regularización con TensorFlow + Keras
**Autor:** Oscar David  
**Curso / CADI:** Percepción Computacional / Redes Neuronales  
**Repositorio:** https://github.com/oscar2121/redes-neuronales-3

### Objetivo:
Implementar la arquitectura optimizada utilizando TensorFlow/Keras con callbacks de EarlyStopping, capas de Dropout, regularizadores L2 y normalización por lotes (BatchNormalization).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Datos
X, y = make_classification(n_samples=3000, n_features=30, n_informative=18, n_redundant=8, n_classes=2, flip_y=0.07, random_state=SEED)
X_tr_raw, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)
X_dev_raw, X_te_raw, y_dev, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=SEED, stratify=y_tmp)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr_raw)
X_dev = scaler.transform(X_dev_raw)
X_te = scaler.transform(X_te_raw)

# Modelo Keras con L2 + BatchNorm + Dropout
model = models.Sequential([
    layers.Input(shape=(30,)),
    layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(), layers.ReLU(), layers.Dropout(0.35),
    
    layers.Dense(64, kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(), layers.ReLU(), layers.Dropout(0.35),
    
    layers.Dense(32, kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(), layers.ReLU(), layers.Dropout(0.25),
    
    layers.Dense(2, activation='softmax')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.003), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)
history = model.fit(X_tr, y_tr, validation_data=(X_dev, y_dev), epochs=75, batch_size=64, callbacks=[early_stop], verbose=1)

# Evaluación
test_loss, test_acc = model.evaluate(X_te, y_te)
print(f'Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}')